# Configuration Check

Confirm that this notebook uses the project configuration before beginning PDF extraction.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "src").is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import CACHE_DIR, GROQ_API_KEY, GROQ_MODEL, PROJECT_ROOT, SAMPLE_DOCS_DIR

assert PROJECT_ROOT == project_root
assert CACHE_DIR.is_relative_to(PROJECT_ROOT)
assert SAMPLE_DOCS_DIR.is_relative_to(PROJECT_ROOT)
assert GROQ_MODEL == "openai/gpt-oss-120b"

print(f"Project root: {PROJECT_ROOT}")
print(f"Groq model: {GROQ_MODEL}")
print(f"Cache directory: {CACHE_DIR}")
print(f"Sample documents directory: {SAMPLE_DOCS_DIR}")
print(f"GROQ_API_KEY configured: {bool(GROQ_API_KEY)}")

Project root: D:\conversIQ_v3
Groq model: openai/gpt-oss-120b
Cache directory: D:\conversIQ_v3\cache\trees
Sample documents directory: D:\conversIQ_v3\data\sample_docs
GROQ_API_KEY configured: True


# PDF Extraction

Extract page-by-page text from a sample PDF using the reusable extraction API.

In [2]:
from src.extraction import ExtractedDocument, extract_pdf

pdf_files = sorted(
    path
    for path in SAMPLE_DOCS_DIR.iterdir()
    if path.is_file() and path.suffix.lower() == ".pdf"
)
if not pdf_files:
    raise FileNotFoundError(
        f"No test PDF found. Place a PDF in {SAMPLE_DOCS_DIR} before running this checkpoint."
    )

pdf_path = pdf_files[0]
document = extract_pdf(pdf_path)

assert isinstance(document, ExtractedDocument)
assert document.filename == pdf_path.name
assert len(document.pages) == document.num_pages
assert [page.page_number for page in document.pages] == list(
    range(1, document.num_pages + 1)
)
assert all(isinstance(page.text, str) for page in document.pages)

print(f"Filename: {document.filename}")
print(f"Number of pages: {document.num_pages}")

for page in document.pages[:3]:
    preview = page.text.strip().replace("\n", " ")[:300]
    print(f"\nPage {page.page_number} preview:\n{preview or '[No extractable text]'}")

print(f"\nFull extracted text for page {document.pages[0].page_number}:\n")
print(document.pages[0].text)

Filename: harry_potter.pdf
Number of pages: 100

Page 1 preview:
[No extractable text]

Page 2 preview:
[No extractable text]

Page 3 preview:
Harry Potter and the Goblet Of Fire     by J. K. Rowling Illustrations by Mary Grandpré         Arthur A. Levine Books An Imprint of Scholastic Press

Full extracted text for page 1:


